In [1]:
import os, sys, platform

def detect_env():
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"
    # Colab sets this module
    try:
        import google.colab  # type: ignore
        return "colab"
    except Exception:
        return "local_or_other"

ENV = detect_env()
print("ENV =", ENV)
print("Python =", sys.version)
print("Platform =", platform.platform())

ENV = local_or_other
Python = 3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.4.4.1)]
Platform = macOS-26.3.1-arm64-arm-64bit


# Encodings

### 1. ASCII (American Standard Code for Information Exchange)
    - In 1963 by American Standard Association
    - includes:
        - latin alphabet (upper & lower)
        - digits 0 - 9
        - common symbols
    - each char has a 3 digit code and a unique byte
    - example:
        - A 065 01000001
    - originally designed as 7 bit system representing 2^7(128) unique characters
    - 8th bit was for error checking, formatting or left unused
    
### 2. Extended ASCII
    - In 1981 by IBM

### 3. The Unicode Standard
    - In 1991 by Unicode Consortium
    - Unicode is a text encoding standard that maps characters to integer code points.
    - introduced to address limitations of 8 bit systems
    - allows 32 bits (4 bytes)
    - can accomodate 4.1 Billion characters
    - covers every language (past/present), extra math symbols, currency signs and emojis
    - back compatible with ASCII as first 128 code points are same as ASCII
    - Currently, Unicode 17.0 (Sep 2025), defined 159, 801 characters across 172 scripts
**Raw strings are generally represented as Unicode Strings**

In [2]:
# converting Unicode character to it's integer representation
ord('🌍')

127757

In [3]:
# converting an integer Unicode code point into a string with the corresponding character
chr(127757)

'🌍'

### What Unicode character does chr(0) return?
Returns the null character

### How does this character’s string representation (\_\_repr\_\_()) differ from its printed representation?
Print renders the character for display, repr shows the string object

### What happens when this character occurs in text?
It is treated as a standard character in python, hence contributing to the length of the string. Not visible while printing

In [4]:
s = "Hello\x00World"
print(f"String: {repr(s)}, Length: {len(s)}")

String: 'Hello\x00World', Length: 11


In [5]:
chr(0)

'\x00'

In [6]:
test_string = "this is a" + chr(0) + " string"
test_string

'this is a\x00 string'

In [7]:
print(test_string)

this is a  string


In [8]:
repr(chr(0))

"'\\x00'"

In [9]:
len(test_string)

17

# Unicode Encodings Schemes

- Maps Unicode Character Code Points to a sequence of Bytes
- While the Unicode standard defines a 'mapping' from characters to code points, encoding schemes provide a practical 'packing' method
- The Unicode standard itself defines three encodings: UTF-8, UTF-16, and UTF-32
- UTF-8 being the dominant encoding for the Internet (more than 98% of all webpages).
    
### 1. UTF-8 
   - More efficient in using bytes
   - variable length encoding = more efficient in using bytes
     - if a char can be represented in 1 byte, it will use only 1 byte, it will use more only if needed
   - back compatible with ASCII
   - self synchronizing
     - first bit of a byte indicates if it is a single byte char or a part of multi byte sequence
   
    | Bytes    | Decimal  | Hex        | Characters |
    | -------- | -------- | ---------- | ---------- |
    | 1 Byte   | 0 - 127  | 0x00 - 0x7F| Standard ASCII                                    |
    | 2 Bytes  | 192 - 223| 0xC2 - 0xDF| Latin character with diatrics, hewbrew, etc       |
    | 3 Bytes  | 224 - 239| 0xE0 - 0xEF| Common Chinese, Japanese and Korean               |
    | 4 Bytes  | 240 - 247| 0xF0 - 0xF4| Emojis and less common, historic or special chars |

### 2. UTF-16
    - Usually 2 bytes, some characters take 4 bytes
    - can be faster for non-latin languages like Chinese/Hindi
    - uses more memory for Enlish compared to UTF-8, as most characters take 2 bytes and some can even take 4
    - not compatible to ASCII

### 3. UTF-32
    - fixed 4 bytes
    - basically just indexing
    - takes a lot of space
    
**It’s impractical to train tokenizers directly on Unicode code points:**
- since the vocabulary would be prohibitively large (around 150K items)
- sparse (since many characters are quite rare)

By converting our Unicode code points into a sequence of bytes (e.g., via the UTF-8 encoding), we are
essentially taking a sequence of code points (21-bit integers with 159,801 valid values) and transforming it
into a sequence of byte values (integers in the range 0 to 255). The 256-length byte vocabulary is much
more manageable to deal with. When using byte-level tokenization, we do not need to worry about out-of-vocabulary tokens, since we know that any input text can be expressed as a sequence of integers from 0
to 255.

In [10]:
test_string = "hello! こんにちは!"

In [11]:
utf8_encoded = test_string.encode("utf-8")
utf8_encoded

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'

In [12]:
type(utf8_encoded)

bytes

In [13]:
print(list(utf8_encoded))

[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [14]:
print(len(test_string))

13


In [15]:
print(len(utf8_encoded))

23


In [16]:
print(utf8_encoded.decode("utf-8"))

hello! こんにちは!


### What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32?

| Feature        |	UTF-8            | UTF-16               | UTF-32                |
| -------        |	-----            | ------               | ------                |
| Base Units     |	1–4 bytes	     | 2 or 4 bytes	        |      Fixed 4 bytes    |
| ASCII Overhead |	0% (1 byte/char) |	100% (2 bytes/char)	| 300% (4 bytes/char)   |
| Initial Vocab  |  256 IDs          |	~65k IDs            | ~1.1M IDs (potential) |

    
### Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

Function is incorrect because we cannot decode individual bytes while using UTF-8. As discussed above, in a multi byte sequnce, the first byte represents if it is a part of multibyte sequence

It will fail for all the chars taking 2 or more bytes

examples:
 - é, ö, ñ.
 - hello! こんにちは!

### Give a two-byte sequence that does not decode to any Unicode character(s).

- 0xC0 0xC1: These lead bytes are prohibited in UTF-8 because they can only be used to create overlong sequences.
- 0xC2 0x00: A sequence where the first byte indicates a two-byte character (ranging from 0xC2 to 0xDF), but the second byte is not a valid continuation byte (which must be between 0x80 and 0xBF).
- 0x80 0x80: A sequence consisting only of continuation bytes without a preceding "lead byte" to start the character. 

In [17]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [18]:
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [19]:
example_string = "é, ö, ñ"
decode_utf8_bytes_to_str_wrong(example_string.encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data

In [20]:
example_string = "hello! こんにちは!"
decode_utf8_bytes_to_str_wrong(example_string.encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data

# Tokenization
- Process of segmenting raw text into discrete units (tokens) and mapping them into integers
- Bridges the gap b/w human redable text and integer sequences that ML models can take as an input
- Tokenizer desgin affects:
    1. Vocab Size
    2. Sequence Length
    3. Ability ot handle rare inputs
    4. Multilingual Faireness
    5. Computational Cost
    6. Downstream Task Performance
    
## Tokenization Strategies

In [21]:
sample_text = (
    "The quick brown fox jumps over the lazy dog. "
    "The dog barked at the fox. The fox ran away quickly. "
    "A quick brown dog is not the same as a lazy fox. "
) * 50

test_sentence = "The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!"

print(f"Training corpus: {len(sample_text)} characters")
print(f"Test sentence: {test_sentence!r}")

Training corpus: 7350 characters
Test sentence: 'The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!'


In [54]:
# Data file paths
if ENV in ("colab", "kaggle"):
    DATA_DIR = '/content/llm-end-to-end/data' 
else:
    DATA_DIR = '../data'

DEBUG_FILE = os.path.join(DATA_DIR, 'TinyStoriesV2-GPT4-valid.txt')
TS_TRAIN_FILE = os.path.join(DATA_DIR, 'TinyStoriesV2-GPT4-train.txt')
TS_VALID_FILE = os.path.join(DATA_DIR, 'TinyStoriesV2-GPT4-valid.txt')
OWT_TRAIN_FILE = os.path.join(DATA_DIR, 'owt_train.txt')
OWT_VALID_FILE = os.path.join(DATA_DIR, 'owt_valid.txt')

In [23]:
def get_compression_ratio(string: str, indices: list[int]) -> float:
    """Given `string` that has been tokenized into `indices`, ."""
    num_bytes = len(bytes(string, encoding="utf-8"))  # @inspect num_bytes
    num_tokens = len(indices)                       # @inspect num_tokens
    return num_bytes / num_tokens

### 1. Bytes based tokenization
- encoding to UTF-8
- any input text can be represented as a sequence of integers from 0 - 255
- no need to worry about out-of-vocabulary tokens
- Vocab Size = 256

In [24]:
from llm_e2e.tokenizer import ByteTokenizer

In [25]:
tokenizer = ByteTokenizer()
indices = tokenizer.encode(test_sentence)  # @inspect indices
reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string
assert test_sentence == reconstructed_string

In [26]:
compression_ratio = get_compression_ratio(test_sentence, indices)
assert compression_ratio == 1

### 2. Character based tokenization
- every char is a token
- sequence becomes very long
- models need to learn spelling, morphology, semantics **which requires** more compute and deep architecture to capture long range dependencies
- example: ByT5
- Vocabulary size: There are approximately 150K Unicode characters.
    - Problem 1: this is a very large vocabulary.
    - Problem 2: many characters are quite rare (e.g., 🌍), which is inefficient use of the vocabulary.

In [27]:
from llm_e2e.tokenizer import CharacterTokenizer

In [28]:
tokenizer = CharacterTokenizer()
indices = tokenizer.encode(test_sentence)  # @inspect indices
reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string
assert test_sentence == reconstructed_string

In [29]:
compression_ratio = get_compression_ratio(test_sentence, indices)
compression_ratio

1.1206896551724137

### 3. Word Level Tokenization
- split on whitespaces and punctuations
- was default in early NLP
- Cons:
 - Vocabulary explosion
 - out-of-vocab tokens: anything not seen during training becomes <unk>
 - no param sharing: run, running, runner, runs are different and completely unlrelated even though they share a root

In [30]:
from llm_e2e.tokenizer import WordTokenizer

In [31]:
tokenizer = WordTokenizer(text=sample_text)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"\nVocabulary:")
for word, idx in sorted(tokenizer.stoi.items(), key=lambda x: x[1]):
    print(f"  {idx}: {word!r}")

Vocab size: 24

Vocabulary:
  0: '<pad>'
  1: '<unk>'
  2: ' ran'
  3: ' dog'
  4: ' over'
  5: ' is'
  6: ' fox'
  7: ' quick'
  8: '.'
  9: 'The'
  10: ' same'
  11: ' away'
  12: ' A'
  13: ' brown'
  14: ' jumps'
  15: ' at'
  16: ' the'
  17: ' not'
  18: ' a'
  19: ' quickly'
  20: ' as'
  21: ' The'
  22: ' barked'
  23: ' lazy'


In [32]:
encoded = tokenizer.encode(test_sentence)
decoded = tokenizer.decode(encoded)

print(f"Original:  {test_sentence!r}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   {decoded!r}")
print(f"Tokens:    {len(encoded)}")
print(f"Unknown tokens (0s): {encoded.count(1)}")
print(f"Compression Ratio: {get_compression_ratio(test_sentence, encoded)}")

Original:  'The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!'
Encoded:   [9, 7, 13, 6, 14, 4, 16, 23, 3, 8, 1, 1, 1, 1, 1, 1]
Decoded:   'The quick brown fox jumps over the lazy dog.<unk><unk><unk><unk><unk><unk>'
Tokens:    16
Unknown tokens (0s): 6
Compression Ratio: 4.0625


In [33]:
GPT2_TOKENIZER_REGEX = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [34]:
tokenizer = WordTokenizer(text=sample_text, pattern=GPT2_TOKENIZER_REGEX)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"\nVocabulary:")
for word, idx in sorted(tokenizer.stoi.items(), key=lambda x: x[1]):
    print(f"  {idx}: {word!r}")

Vocab size: 25

Vocabulary:
  0: '<pad>'
  1: '<unk>'
  2: ' ran'
  3: ' dog'
  4: ' over'
  5: ' is'
  6: ' fox'
  7: ' quick'
  8: '.'
  9: 'The'
  10: ' same'
  11: ' away'
  12: ' A'
  13: ' brown'
  14: ' jumps'
  15: ' at'
  16: ' the'
  17: ' not'
  18: ' a'
  19: ' quickly'
  20: ' as'
  21: ' '
  22: ' The'
  23: ' barked'
  24: ' lazy'


In [35]:
encoded = tokenizer.encode(test_sentence)
decoded = tokenizer.decode(encoded)

print(f"Original:  {test_sentence!r}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   {decoded!r}")
print(f"Tokens:    {len(encoded)}")
print(f"Unknown tokens (0s): {encoded.count(1)}")
print(f"Compression Ratio: {get_compression_ratio(test_sentence, encoded)}")

Original:  'The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!'
Encoded:   [9, 7, 13, 6, 14, 4, 16, 24, 3, 8, 1, 1, 1, 1, 1]
Decoded:   'The quick brown fox jumps over the lazy dog.<unk><unk><unk><unk><unk>'
Tokens:    15
Unknown tokens (0s): 5
Compression Ratio: 4.333333333333333


### 4. N-gram Tokenization
- continuous sequence of n words or characters
- Word N-grams for "the cat sat on the wall":
    1. Unigram: [the, cat, sat, on, the, wall]
    2. Bigram: [the cat, cat sat, sat on, on the, the wall]
    3. Trigram: [ the cat sat, cat sat on, sat on the, on the wall]
- Char N-grams for "hello":
    1. Unigram: [h, e, l, l, o]
    2. Bigram: [he, el, ll, lo]
    3. Trigram: [hel, ell, llo]
    4. Trigram with end tags: [\<he, hel, ell, llo, lo\>]
- Vocabulary grows combinatorially

### 5. Subword Tokenization
- keeps frequesnt words as whole
- splits rare words into meaningful subword pieces
- gives a fixed vocab
- can represent any possible input
- no out-of-vocab tokens
- shares params across morpholigically related words

It is a midpoint b/w word-level and byte-level tokenization:
- byte-level tokenizer has 256 entried in the vocab
- a subword tokenizer trades off larges vocab size for better compression

Example: If byte sequence b'the' occurs very often in training data, it is assigned a new token in the vocabulary
- vocab size will become 257
- b'the' becomes 1 token instead of a 3 token sequence

## Subword Tokenization Algorithms

### 1. Byte Pair Encoding (BPE)
- Introduced in 1994 by Philip Gage for **Data Compression**
- In 2015, it was adapted to NLP for neural machine translation. Previously, word based tokenization was used
- BPE was then used by GPT-2

- Basic idea: train the tokenizer on raw text to automatically determine the vocabulary.
- Intuition: common sequences of characters are represented by a single token, rare sequences are represented by many tokens.
- The GPT-2 paper used word-based tokenization to break up the text into inital segments and run the original BPE algorithm on each segment.
- Sketch: start with each byte as a token, and successively merge the most common pair of adjacent tokens.

#### BPE Tokenizer Training
- **Step 1: Vocabulary initialization**
    - The tokenizer vocabulary is a one-to-one mapping from bytestring token to integer ID. 
    - Since we’re training a byte-level BPE tokenizer, our initial vocabulary is simply the set of all bytes.
    - Since there are 256 possible byte values, our initial vocabulary is of size 256.
- **Step 2: Pre-tokenization**
    - Once we have a vocabulary, in principle, we can count how often bytes occur next to each other in the text
    - Begin merging them starting with the most frequent pair of bytes. 
    - However, this is quite computationally expensive, since we’d have to take a full pass over the corpus each time we merge.
    - In addition, directly merging bytes across the corpus may result in tokens that differ only in punctuation (e.g., dog! vs. dog.). 
    - These tokens would get completely different token IDs, even though they are likely to have high semantic similarity (since they differ only in punctuation).
    - To avoid this, we **pre-tokenize** the corpus.
    - We can think of this as a coarse-grained tokenization over the corpus that helps us count how often pairs of characters appear. 
    - For example, the word 'text' might be a pre-token that appears 10 times. 
    - In this case, when we count how often the characters ‘t’ and ‘e’ appear next to each other
    - we will see that the word ‘text’ has ‘t’ and ‘e’ adjacent and we can increment their count by 10 instead of looking through the corpus. 
    - Since we’re training a byte-level BPE model, each pretoken is represented as a sequence of UTF-8 bytes.
    - The original BPE implementation pre-tokenizes by simply splitting on whitespace (i.e., s.split(" ")). 
    - This method is still found in tokenizers based on SentencePiece (for instance the Llama 1 and 2 tokenizer).
    - Most modern tokenizers use a regex-based pre-tokenizer, a practice from GPT-2
- **Step 3: Compute BPE Merges**
    - Now that we’ve converted our input text into pre-tokens and represented each pre-token as a sequence of UTF-8 bytes
    - we can compute the BPE merges (i.e., train the BPE tokenizer). 
    - At a high level, the BPE algorithm iteratively counts every pair of bytes and identifies the pair with the highest frequency (“A”,“B”). 
    - Every occurrence of this most frequent pair (“A”, “B”) is then merged, i.e., replaced with a new token “AB”. 
    - This new merged token is added to our vocabulary; 
    - As a result, the final vocabulary after BPE training is the size of the initial vocabulary (256 in our case), plus the number of BPE merge operations performed during training. 
    - For efficiency during BPE training, we do not consider pairs that cross pre-token boundaries like end-of-word token
    - When computing merges, deterministically break ties in pair frequency by preferring the lexicographically greater pair. 
    - For example, if the pairs (“A”, “B”), (“A”, “C”), (“B”, “ZZ”), and (“BA”, “A”) all have the highest frequency, we’d merge (“BA”, “A”)
- **Special Tokens**
    - Often, some strings (e.g., <|endoftext|>) are used to encode metadata (e.g., boundaries between documents). 
    - When encoding text, it’s often desirable to treat some strings as “special tokens” that should never be split into multiple tokens (i.e., will always be preserved as a single token). 
    - For example, the endof-sequence string <|endoftext|> should always be preserved as a single token (i.e., a single integer ID), so we know when to stop generating from the language model. 
    - These special tokens must be added to the vocabulary, so they have a corresponding fixed token ID.

#### BPE Tokenizer: Basic Implementation

In [36]:
from llm_e2e.tokenizer import BPEBasicTokenizer

In [37]:
import sys
import time
sys.path.insert(0, '.')

In [38]:
merge_counts = [0, 10, 50, 100, 200, 500]

print(f"{'Merges':<10} {'Vocab Size':<12} {'Tokens':<10} {'Compression':<15} {'Train Time':<12}")
print("-" * 60)

for n in merge_counts:
    start = time.time()
    bpe = BPEBasicTokenizer(text=sample_text, num_merges=n)
    train_time = time.time() - start
    
    encoded = bpe.encode(test_sentence)
    raw_bytes = len(test_sentence.lower().encode('utf-8'))
    compression = f"{raw_bytes / len(encoded):.2f}x" if encoded else "N/A"
    
    print(f"{n:<10} {bpe.vocab_size:<12} {len(encoded):<10} {compression:<15} {train_time:.3f}s")

Merges     Vocab Size   Tokens     Compression     Train Time  
------------------------------------------------------------
0          256          65         1.00x           0.002s
10         266          55         1.18x           0.002s
50         306          34         1.91x           0.003s
100        316          31         2.10x           0.005s
200        316          31         2.10x           0.003s
500        316          31         2.10x           0.003s


In [39]:
# Inspect what BPE learned
corpus = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"

bpe = BPEBasicTokenizer(
    text=corpus,
    num_merges=6,
)

print(f"Vocab size: {bpe.vocab_size}")

print("First 6 BPE merges learned:")
print(f"{'Merge #':<10} {'Pair':<20} {'New Token':<15} {'New ID':<10}")
print("-" * 55)
for i, (pair, new_id) in enumerate(bpe.bpe_merges.items()):
    left = bpe.vocab[pair[0]]
    right = bpe.vocab[pair[1]]
    merged = bpe.vocab[new_id]
    print(f"{i+1:<10} ({pair[0]}, {pair[1]}){'':>8} {merged!r:<15} {new_id}")

Vocab size: 262
First 6 BPE merges learned:
Merge #    Pair                 New Token       New ID    
-------------------------------------------------------
1          (115, 116)         b'st'           256
2          (101, 256)         b'est'          257
3          (111, 119)         b'ow'           258
4          (108, 258)         b'low'          259
5          (119, 257)         b'west'         260
6          (110, 101)         b'ne'           261


In [40]:
# Detailed encode/decode
bpe = BPEBasicTokenizer(text=sample_text, num_merges=100)

encoded = bpe.encode(test_sentence)
decoded = bpe.decode(encoded)

print(f"Original:  {test_sentence!r}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   {decoded!r}")
print(f"Tokens:    {len(encoded)}")
print(f"Roundtrip: {'PASS' if decoded == test_sentence else 'FAIL'}")

Original:  'The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!'
Encoded:   [268, 270, 277, 259, 310, 312, 271, 281, 269, 46, 32, 72, 101, 108, 108, 111, 44, 32, 240, 159, 140, 141, 33, 32, 228, 189, 160, 229, 165, 189, 33]
Decoded:   'The quick brown fox jumps over the lazy dog. Hello, 🌍! 你好!'
Tokens:    31
Roundtrip: PASS


In [41]:
# BPE with Special Tokens
special_tokens = ["<|endoftext|>", "<|pad|>"]
corpus = "hello world<|endoftext|>goodbye world<|endoftext|>the end"

bpe_special = BPEBasicTokenizer(
    text=corpus,
    special_tokens=special_tokens,
    num_merges=50,
)

print(f"Special tokens: {bpe_special.special_tokens}")
print(f"Vocab size: {bpe_special.vocab_size}")

test = "hello<|endoftext|>goodbye"
encoded = bpe_special.encode(test)
decoded = bpe_special.decode(encoded)

print(f"\nOriginal:  {test!r}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   {decoded!r}")
print(f"Roundtrip: {'PASS' if decoded == test else 'FAIL'}")

Special tokens: {'<|endoftext|>': 256, '<|pad|>': 257}
Vocab size: 277

Original:  'hello<|endoftext|>goodbye'
Encoded:   [266, 256, 274]
Decoded:   'hello<|endoftext|>goodbye'
Roundtrip: PASS


#### BPETokenizer vs WordTokenizer

In [42]:
test_texts = [
    "The quick brown fox",
    "an elephant danced gracefully",  # unseen words
    "hello, world! how's it going?",  # punctuation + contractions
    "café résumé naïve",              # unicode
]

word_tok = WordTokenizer(text=sample_text)
bpe_tok = BPEBasicTokenizer(text=sample_text, num_merges=200)

print(f"{'Text':<35} {'Word Tokens':<15} {'BPE Tokens':<15} {'Word Roundtrip':<15} {'BPE Roundtrip'}")
print("-" * 95)

for text in test_texts:
    word_enc = word_tok.encode(text)
    bpe_enc = bpe_tok.encode(text)

    word_dec = word_tok.decode(word_enc)
    bpe_dec = bpe_tok.decode(bpe_enc)
    
    word_rt = word_dec == text
    bpe_rt = bpe_dec == text

    print(f"{text:<35} {len(word_enc):<15} {len(bpe_enc):<15} {'PASS' if word_rt else 'FAIL':<15} {'PASS' if bpe_rt else 'FAIL'}")

Text                                Word Tokens     BPE Tokens      Word Roundtrip  BPE Roundtrip
-----------------------------------------------------------------------------------------------
The quick brown fox                 4               4               PASS            PASS
an elephant danced gracefully       4               28              FAIL            PASS
hello, world! how's it going?       10              28              FAIL            PASS
café résumé naïve                   3               19              FAIL            PASS


In [43]:
# speed
iterations = 500

# Word tokenizer
start = time.time()
for _ in range(iterations):
    word_tok.encode(test_sentence)
word_time = time.time() - start

# BPE tokenizer
start = time.time()
for _ in range(iterations):
    bpe_tok.encode(test_sentence)
bpe_time = time.time() - start

print(f"Encoding {iterations} iterations:")
print(f"  Word Tokenizer: {word_time:.3f}s ({word_time/iterations*1000:.2f}ms per encode)")
print(f"  BPE Tokenizer:  {bpe_time:.3f}s ({bpe_time/iterations*1000:.2f}ms per encode)")
print(f"  BPE is {bpe_time/word_time:.1f}x slower than Word")

Encoding 500 iterations:
  Word Tokenizer: 0.010s (0.02ms per encode)
  BPE Tokenizer:  0.144s (0.29ms per encode)
  BPE is 14.3x slower than Word


#### BPE Tokenizer: Optimizing the implementation

In [44]:
from llm_e2e.tokenizer import BPEOptimTokenizer

In [45]:
SPECIAL_TOKENS = ["<|endoftext|>"]
MAX_VOCAB = 10000
NUM_MERGES = MAX_VOCAB - 256 - len(SPECIAL_TOKENS) 
GPT2_TOKENIZER_REGEX = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


##### Step 1: Parallelizing the Pretokenization part using multiprocessing
##### Comparing with Serial Pretokenization

In [46]:
basic = BPEBasicTokenizer(pattern=GPT2_TOKENIZER_REGEX, special_tokens=SPECIAL_TOKENS)
optim = BPEOptimTokenizer(pattern=GPT2_TOKENIZER_REGEX, special_tokens=SPECIAL_TOKENS)

with open(DEBUG_FILE) as f:
    text = f.read()
parts = basic.split_on_special_tokens(text)

t0 = time.time()
basic.pretokenize(parts)
t_basic = time.time() - t0

t0 = time.time()
optim.pretokenize_optim(parts)
t_optim = time.time() - t0

print(f"Serial:   {t_basic:.2f}s")
print(f"Parallel: {t_optim:.2f}s")
print(f"Speedup:  {t_basic / t_optim:.2f}x")

Serial:   1.87s
Parallel: 1.34s
Speedup:  1.40x


##### Step 2: Optimizing the Merging step
The core idea: 
- instead of recomputing pair_freq from scratch every iteration, we can build it once and update it incrementally after each merge.
- using max_heap instead of scanning all pairs every iteration. Replacing with a max-heap gives O(log V) per iteration instead of O(V).

After merging (a, b) → c, only the pairs adjacent to where the merge happened change. Everything else in the corpus is untouched.

What changes after merging (a, b) → c in a word like [... x, a, b, y ...]:

- (x, a) disappears → (x, c) appears
- (b, y) disappears → (c, y) appears
- (a, b) itself disappears entirely

So we need two data structures:

1. pair_freq — the current pair counts (updated incrementally, not rebuilt)
2. pair_to_words — an inverted index mapping each pair → set of words that contain it (so after a merge we know which words to look at)

The updated train loop becomes:

1. Build pair_freq, pair_to_words and max_heap once before the loop
2. Each iteration: pick max(pair_freq) from the heap
3. For each word containing the merged pair:
    - Walk through the word, find positions of (a, b)
    - For each occurrence, decrement (x, a) and (b, y), increment (x, c) and (c, y)
    - Update pair_to_words — remove old pairs' references to this word, add new ones
    - Update the heap
4. Delete (a, b) from pair_freq
5. Update byte_word_freq as before (still need it for pair_to_words lookups)

##### Comparing with basic BPE Implementation 

In [47]:
times = []
for cls, label in [(BPEBasicTokenizer, "Basic"), (BPEOptimTokenizer, "Optim")]:
    t0 = time.time()
    tok = cls.from_file(
        filepath=DEBUG_FILE,
        pattern=GPT2_TOKENIZER_REGEX,
        special_tokens=SPECIAL_TOKENS,
        num_merges=NUM_MERGES,
    )
    print(f"{label}: {time.time() - t0:.1f}s | merges={len(tok.bpe_merges)}")
    times.append((label, time.time() - t0))

speedup = times[0][1] / times[1][1]
print(f"Speedup: {speedup:.2f}x")

basic = BPEBasicTokenizer(text=sample_text, num_merges=100)
optim = BPEOptimTokenizer(text=sample_text, num_merges=100)

assert list(basic.bpe_merges.keys()) == list(optim.bpe_merges.keys()), "Merge order mismatch!"
assert basic.encode(test_sentence) == optim.encode(test_sentence), "Encode mismatch!"
print("PASS")

Basic: 101.6s | merges=9743
Optim: 2.2s | merges=9743
Speedup: 45.41x
PASS


#### BPE Tokenizer: Experiments
##### BPE Training on Tiny Stories

In [48]:
# import cProfile
# import pstats
# import io

# pr = cProfile.Profile()
# pr.enable()

BPE_TS = BPEOptimTokenizer.from_file(
    filepath=TS_TRAIN_FILE,
    pattern=GPT2_TOKENIZER_REGEX,
    special_tokens=SPECIAL_TOKENS,
    num_merges=NUM_MERGES,
)

# pr.disable()

# stream = io.StringIO()
# stats = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
# stats.print_stats(20)
# print(stream.getvalue())

In [49]:
BPE_TS.save(os.path.join(DATA_DIR, 'bpe_ts_train_10k.json'))

# next time, load in seconds instead of minutes
# BPE_TS = BPEOptimTokenizer.from_pretrained(os.path.join(DATA_DIR, 'bpe_ts_train_10k.json'))

##### Compression Ratio
On a sample of 10 documents from TinyStories

In [50]:
with open(TS_TRAIN_FILE) as f:
    raw = f.read()

# documents are separated by <|endoftext|>
docs = [d.strip() for d in raw.split("<|endoftext|>") if d.strip()]
sample_docs = docs[:10]

for i, doc in enumerate(sample_docs):
    print(f"--- Doc {i+1} ({len(doc)} chars) ---")
    print(doc[:200])
    print()

--- Doc 1 (730 chars) ---
Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking

--- Doc 2 (661 chars) ---
Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
One day, Ollie's mom said, "Ollie, hurry and get some fish for 

--- Doc 3 (513 chars) ---
One day, a little boy named Tim went to the park. He saw a big tiger. The tiger was not mean, but very easy to play with. Tim and the tiger played all day. They had lots of fun.
Then, something unexpe

--- Doc 4 (856 chars) ---
Once upon a time there was a friendly little boy called Bob. Bob loved to pick flowers and look for birds. One day he decided to go outside with his friends to pick some more flowers. 
He suddenly not

--- Doc 5 (954 chars) ---
Once upon a time, in a small house, there lived a little girl 

In [51]:
ratios = []
for doc in sample_docs:
    ids = BPE_TS.encode(doc)
    ratio = get_compression_ratio(doc, ids)
    ratios.append(ratio)
    print(f"  {len(doc):>6} bytes → {len(ids):>5} tokens  ratio={ratio:.2f}")

print(f"\nMean compression ratio: {sum(ratios)/len(ratios):.2f} bytes/token")

     730 bytes →   199 tokens  ratio=3.71
     661 bytes →   193 tokens  ratio=3.42
     513 bytes →   137 tokens  ratio=3.74
     856 bytes →   221 tokens  ratio=3.87
     954 bytes →   295 tokens  ratio=3.23
     678 bytes →   196 tokens  ratio=3.46
     624 bytes →   174 tokens  ratio=3.59
     439 bytes →   118 tokens  ratio=3.72
    1079 bytes →   342 tokens  ratio=3.15
     870 bytes →   234 tokens  ratio=3.72

Mean compression ratio: 3.56 bytes/token


##### Throughput Estimation (bytes/second)

In [52]:
# build a ~1 MB chunk from samples
bench_text = " ".join(sample_docs * 20)
bench_bytes = len(bench_text.encode("utf-8"))

# warm up
BPE_TS.encode(bench_text[:1000])

# measure
t0 = time.time()
_ = BPE_TS.encode(bench_text)
elapsed = time.time() - t0

throughput_bytes_per_sec = bench_bytes / elapsed
throughput_mb_per_sec = throughput_bytes_per_sec / 1e6

print(f"Input size : {bench_bytes/1e6:.2f} MB")
print(f"Time       : {elapsed:.3f}s")
print(f"Throughput : {throughput_mb_per_sec:.2f} MB/s")

Input size : 0.15 MB
Time       : 15.162s
Throughput : 0.01 MB/s


##### Estimated Time to train on OWT Dataset

In [55]:
owt_size_bytes = os.path.getsize(OWT_TRAIN_FILE)

estimated_seconds = owt_size_bytes / throughput_bytes_per_sec
estimated_hours = estimated_seconds / 3600

print(f"OWT TRAIN size      : {owt_size_bytes/1e9:.0f} GB")
print(f"Throughput    : {throughput_mb_per_sec:.2f} MB/s")
print(f"Estimated time: {estimated_hours:.1f} hours")

OWT TRAIN size      : 12 GB
Throughput    : 0.01 MB/s
Estimated time: 338.2 hours


##### Encoding and Serializing Tokens

In [ ]:
import numpy as np
import os

def encode_and_save(tokenizer, filepath, out_path):
    with open(filepath) as f:
        text = f.read()
    ids = tokenizer.encode(text)
    arr = np.array(ids, dtype=np.uint16)
    np.save(out_path, arr)
    print(f"{os.path.basename(filepath)}: {len(ids):,} tokens → {arr.nbytes / 1e6:.1f} MB")

encode_and_save(BPE_TS, TS_TRAIN_FILE, os.path.join(DATA_DIR, 'TS_TRAIN.npy'))
encode_and_save(BPE_TS, TS_VALID_FILE, os.path.join(DATA_DIR, 'TS_VALID.npy'))